## nb41 — CD5 DiD: Awardees vs Non-Awardees (±5 Years)

Re-analysis of nb29 CD5 trajectories reframed as a proper DiD:
- **Treatment** (`is_awardee=1`): authors in the `award` group from nb28 panel
- **Control** (`is_awardee=0`): authors in the `control` group
- **Outcome**: paper-level CD5 score, aggregated to author-year level
- **Event study**: relative year −5 to +5, reference year = −1
- **DiD OLS**: `cd5 ~ C(rel_year) * is_awardee + C(author_id) + C(paper_year)` (TWFE)

All data reused from `data/cd_trajectory/` — no new API calls needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
from pathlib import Path
from scipy import stats

ROOT    = Path('..')
CD_DIR  = ROOT / 'data' / 'cd_trajectory'
FIG_DIR = ROOT / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

### 1. Load & merge panel + CD5 scores

In [ ]:
panel = pd.read_csv(CD_DIR / 'author_papers_panel.csv')
cd5   = pd.read_csv(CD_DIR / 'paper_cd5_scores.csv')

print('Panel shape:', panel.shape)
print('CD5 scores shape:', cd5.shape)
print('Panel columns:', panel.columns.tolist())
print('CD5 columns:', cd5.columns.tolist())

In [ ]:
# map group -> is_awardee binary
panel['is_awardee'] = (panel['group'] == 'award').astype(int)

# merge CD5
merged = panel.merge(cd5[['paper_id', 'cd5']], on='paper_id', how='inner')
print(f'After CD5 merge: {len(merged):,} rows  |  coverage: {len(merged)/len(panel)*100:.1f}%')
print(merged.groupby('is_awardee')['author_id'].nunique().rename({0:'control', 1:'awardee'}))

### 2. Author-year aggregation

In [ ]:
# keep ±5 window
merged = merged[merged['relative_year'].between(-5, 5)]

author_year = (
    merged
    .groupby(['author_id', 'is_awardee', 'award_year', 'relative_year', 'paper_year'])
    ['cd5'].mean()
    .reset_index()
    .rename(columns={'cd5': 'mean_cd5', 'paper_year': 'cal_year'})
)
print('Author-year panel:', author_year.shape)
print(author_year.groupby('is_awardee')['author_id'].nunique())

### 3. Descriptive trajectory — group means ± 95% CI

In [ ]:
def ci95(x):
    n = len(x)
    if n < 2: return np.nan
    se = stats.sem(x, nan_policy='omit')
    return 1.96 * se

traj = (
    author_year
    .groupby(['is_awardee', 'relative_year'])['mean_cd5']
    .agg(mean='mean', ci=ci95, n='count')
    .reset_index()
)
print(traj)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = {1: '#2c7bb6', 0: '#d7191c'}
labels = {1: 'Awardees', 0: 'Non-awardees (control)'}

for grp, grp_df in traj.groupby('is_awardee'):
    grp_df = grp_df.sort_values('relative_year')
    ax.plot(grp_df['relative_year'], grp_df['mean'], marker='o', color=colors[grp],
            label=labels[grp], lw=2)
    ax.fill_between(grp_df['relative_year'],
                    grp_df['mean'] - grp_df['ci'],
                    grp_df['mean'] + grp_df['ci'],
                    alpha=0.15, color=colors[grp])

ax.axvline(0, color='gray', lw=1.2, ls='--', label='Award year (t=0)')
ax.axhline(0, color='black', lw=0.6, ls=':')
ax.set_xlabel('Years relative to award year', fontsize=11)
ax.set_ylabel('Mean CD5 score', fontsize=11)
ax.set_title('CD5 Trajectories: Awardees vs Non-Awardees (±5 Years)', fontsize=13)
ax.legend(fontsize=10)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(FIG_DIR / 'cd5_trajectory_awardee_vs_nonawardee.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved trajectory plot.')

### 4. Two-Way Fixed Effects (TWFE) DiD — Event Study

In [ ]:
# reference period = rel_year -1  (drop it so coefficients are relative to t=-1)
reg_df = author_year[author_year['relative_year'] != -1].copy()
reg_df['rel_year_str'] = 'y' + reg_df['relative_year'].astype(str).str.replace('-', 'm')
reg_df['author_fe']    = reg_df['author_id'].astype('category')
reg_df['year_fe']      = reg_df['cal_year'].astype('category')

# event-study: interaction of rel_year dummies with is_awardee
# absorb author + calendar year FE via dummies (small enough panel)
formula = 'mean_cd5 ~ C(rel_year_str) * C(is_awardee) + C(author_fe) + C(year_fe)'
model   = smf.ols(formula, data=reg_df).fit(cov_type='HC3')
print(model.summary().tables[0])

In [ ]:
# extract interaction coefficients  C(rel_year_str)[T.yX]:C(is_awardee)[T.1]
params = model.params
conf   = model.conf_int()

interact_keys = [k for k in params.index if 'rel_year_str' in k and 'is_awardee' in k]

def key_to_yr(k):
    # e.g. "C(rel_year_str)[T.ym3]:C(is_awardee)[T.1]"  -> -3
    part = k.split('[T.')[1].split(']')[0]   # 'ym3' or 'y3'
    return int(part.replace('ym', '-').replace('y', ''))

eventstudy = pd.DataFrame({
    'rel_year': [key_to_yr(k) for k in interact_keys],
    'coef':     [params[k]    for k in interact_keys],
    'ci_lo':    [conf.loc[k, 0] for k in interact_keys],
    'ci_hi':    [conf.loc[k, 1] for k in interact_keys],
}).sort_values('rel_year').reset_index(drop=True)

# add the reference period as zero
ref_row = pd.DataFrame({'rel_year': [-1], 'coef': [0.0], 'ci_lo': [0.0], 'ci_hi': [0.0]})
eventstudy = pd.concat([eventstudy, ref_row], ignore_index=True).sort_values('rel_year').reset_index(drop=True)
print(eventstudy)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

pre  = eventstudy[eventstudy['rel_year'] < 0]
post = eventstudy[eventstudy['rel_year'] >= 0]

for subset, color, label in [(pre, '#888', 'Pre-award'), (post, '#2c7bb6', 'Post-award')]:
    ax.errorbar(subset['rel_year'], subset['coef'],
                yerr=[subset['coef']-subset['ci_lo'], subset['ci_hi']-subset['coef']],
                fmt='o', color=color, capsize=4, lw=1.8, ms=6, label=label)

ax.axvline(0,   color='gray',  lw=1.2, ls='--', label='Award year (t=0)')
ax.axhline(0,   color='black', lw=0.8, ls=':')
ax.set_xlabel('Years relative to award year', fontsize=11)
ax.set_ylabel('DiD coefficient (vs t=−1)', fontsize=11)
ax.set_title('Event Study: Effect of Award on CD5 Score\n(TWFE, author + year FE, HC3 SE)', fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cd5_eventstudy_did.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved event-study plot.')

### 5. Simple 2x2 DiD estimate (pre/post × treated/control)

In [ ]:
author_year['post'] = (author_year['relative_year'] >= 0).astype(int)

simple_did = smf.ols(
    'mean_cd5 ~ post * is_awardee + C(author_id) + C(cal_year)',
    data=author_year
).fit(cov_type='HC3')

coef   = simple_did.params['post:is_awardee']
pval   = simple_did.pvalues['post:is_awardee']
ci_lo, ci_hi = simple_did.conf_int().loc['post:is_awardee']

print(f'DiD estimate (post×awardee): {coef:.4f}')
print(f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'p-value: {pval:.4f}')

### 6. Pre-trend test (parallel trends assumption)

In [ ]:
pre_df = author_year[author_year['relative_year'] < 0].copy()

pretrend = smf.ols(
    'mean_cd5 ~ relative_year * is_awardee + C(author_id) + C(cal_year)',
    data=pre_df
).fit(cov_type='HC3')

pretrend_coef = pretrend.params['relative_year:is_awardee']
pretrend_p    = pretrend.pvalues['relative_year:is_awardee']
pretrend_ci   = pretrend.conf_int().loc['relative_year:is_awardee']

print(f'Pre-trend interaction (slope diff): {pretrend_coef:.4f}')
print(f'95% CI: {pretrend_ci.values}')
print(f'p-value: {pretrend_p:.4f}')
if pretrend_p > 0.05:
    print('✓ Pre-trends not significantly different (parallel trends plausible)')
else:
    print('⚠ Significant pre-trend difference — parallel trends may be violated')

### 7. Summary table

In [ ]:
summary = (
    author_year
    .assign(period=lambda d: d['relative_year'].apply(lambda x: 'pre' if x < 0 else 'post'))
    .groupby(['is_awardee', 'period'])['mean_cd5']
    .agg(['mean', 'median', 'std', 'count'])
    .round(4)
)
print(summary)
summary.to_csv(CD_DIR / 'cd5_did_summary.csv')
print('\nSaved summary to data/cd_trajectory/cd5_did_summary.csv')